# Kiên Đoàn TTS — Ngọc Huyền

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Audio tự động tải về máy (MP3) + hiển thị trên giao diện
4. Bấm **📦 Tải tất cả audio** để tải zip toàn bộ

### Tối ưu đã áp dụng
- ⚡ Speed 1.25x (giọng đọc nhanh hơn 25%)
- ⚡ `num_step=16` (inference nhanh ~2x so với 32)
- ⚡ FlashInfer GPU kernel acceleration
- 📥 Tự động tải audio MP3 về máy sau mỗi lần tạo
- 🎵 Hiển thị audio mới nhất trên giao diện
- 📦 Nút tải toàn bộ audio đã tạo (zip)

> Powered by OmniVoice (Apache 2.0) — github.com/k2-fsa/OmniVoice


In [ ]:
# ==========================================
# Cell 1: Cài đặt dependencies
# ==========================================
print('🔧 Đang cài đặt dependencies (~1-2 phút)...')

# Core: OmniVoice + Gradio + numpy compat
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"

# Audio processing: pydub cho xuất MP3
!pip install -q pydub soundfile

# ffmpeg cho pydub MP3 export
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

# FlashInfer acceleration (~2-3x speedup)
import subprocess, sys
try:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', 'flashinfer-python==0.6'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    print('✅ FlashInfer installed')
except Exception:
    print('⚠️ FlashInfer không cài được (vẫn chạy OK, chỉ chậm hơn một chút)')

# Tải giọng mẫu Ngọc Huyền (self-hosted — repo của mình)
!wget -q https://raw.githubusercontent.com/vinh-nd2002/tts/main/ngoc-huyen.mp3 -O voice_sample.mp3

print('✅ Cài đặt hoàn tất!')
print('✅ Đã tải voice sample Ngọc Huyền!')

In [ ]:
# ==========================================
# Cell 2: Khởi động model + Giao diện TTS
# ==========================================
print('🚀 Đang khởi động OmniVoice... (lần đầu ~3-5 phút, lần sau ~30 giây)')

import logging, os, re, time, datetime, shutil
import numpy as np
import torch
import soundfile as sf
import gradio as gr

# ── Compatibility patches ──────────────────────────────────────────
# Patch: torch._utils removed in torch 2.13+ — alias to torch._C._utils
import torch as _torch
if not hasattr(_torch, '_utils'):
    _torch._utils = _torch._C._utils

# Shim: AutoFeatureExtractor removed in transformers 5.x
import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# ── GPU Detection ─────────────────────────────────────────────────
for i in range(30):
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
        print(f'🖥️ GPU: {gpu_name} ({gpu_mem:.1f} GB)')
        break
    time.sleep(1)
else:
    print('❌ GPU not available — Runtime > Change runtime type > T4 GPU')

# ── Load Model ─────────────────────────────────────────────────────
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {DEVICE}...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
logger.info(f'✅ Model ready — SR: {SAMPLING_RATE}Hz')

# ── Voice Clone Prompt ─────────────────────────────────────────────
logger.info('Creating VoiceClonePrompt...')
VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio='voice_sample.mp3')
logger.info('✅ Voice prompt ready — Ngọc Huyền')

# ── Generation Config (OPTIMIZED) ──────────────────────────────────
# num_step=16: ~2x faster than 32, quality still excellent
# speed=1.25: giọng đọc nhanh hơn 25%
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=16,             # ⚡ Giảm từ 32 → 16
    guidance_scale=1.8,
    denoise=True,
    preprocess_prompt=True,
    postprocess_output=True,
    position_temperature=5.0,
    class_temperature=0.2,
    pad_duration=0.1,
    fade_duration=0.1,
)

SPEAK_SPEED = 1.25  # ⚡ Tốc độ nói 1.25x

# ── Output Directory ──────────────────────────────────────────────
OUTPUT_DIR = '/content/audio_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Track generated files
audio_history = []

# ── Auto Download Helper ──────────────────────────────────────────
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def save_audio_mp3(audio_np, sr):
    """Lưu audio thành MP3 + tự động tải về máy nếu đang trên Colab."""
    from pydub import AudioSegment
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    wav_path = os.path.join(OUTPUT_DIR, f'ngoc_huyen_{ts}.wav')
    mp3_path = os.path.join(OUTPUT_DIR, f'ngoc_huyen_{ts}.mp3')

    # Save WAV first
    sf.write(wav_path, audio_np, sr)

    # Convert to MP3 (192kbps, high quality)
    AudioSegment.from_wav(wav_path).export(mp3_path, format='mp3', bitrate='192k')
    os.remove(wav_path)  # Clean up WAV

    audio_history.append(mp3_path)
    logger.info(f'💾 Saved: {mp3_path}')

    # Auto download on Colab
    if IN_COLAB:
        try:
            colab_files.download(mp3_path)
            logger.info(f'📥 Auto download triggered: {os.path.basename(mp3_path)}')
        except Exception as e:
            logger.warning(f'Auto download failed (dùng share link?): {e}')

    return mp3_path

# ── Generate Function ─────────────────────────────────────────────
@torch.inference_mode()
def generate_voice(text: str):
    """Generate voice from text with speed 1.25x."""
    text = text.strip()
    if not text:
        return None, None, 'Vui lòng nhập văn bản!'

    start_time = time.time()

    # Split paragraphs
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

    if len(paragraphs) == 1:
        audio = model.generate(
            text=paragraphs[0],
            voice_clone_prompt=VOICE_PROMPT,
            language='vi',
            speed=SPEAK_SPEED,
            generation_config=GEN_CFG
        )[0]
    else:
        audios = []
        for i, p in enumerate(paragraphs):
            a = model.generate(
                text=p,
                voice_clone_prompt=VOICE_PROMPT,
                language='vi',
                speed=SPEAK_SPEED,
                generation_config=GEN_CFG
            )[0]
            audios.append(a)
            if i < len(paragraphs) - 1:
                # 0.3s silence between paragraphs
                audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
        audio = np.concatenate(audios)

    elapsed = time.time() - start_time
    duration = len(audio) / SAMPLING_RATE

    # Save MP3 + auto download
    mp3_path = save_audio_mp3(audio, SAMPLING_RATE)

    # Prepare audio for Gradio player
    waveform = (audio * 32767).astype(np.int16)

    status = (
        f'✅ Tạo thành công! '
        f'⏱️ {elapsed:.1f}s | '
        f'🎵 {duration:.1f}s audio | '
        f'📥 {os.path.basename(mp3_path)} | '
        f'📁 Tổng: {len(audio_history)} file'
    )

    return (SAMPLING_RATE, waveform), mp3_path, status

# ── Download All Function ─────────────────────────────────────────
def download_all_audio():
    """Nén tất cả audio → zip → tải về."""
    if not audio_history:
        return None, '❌ Chưa có audio nào để tải!'

    zip_path = '/content/all_audio'
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    zip_file = f'{zip_path}.zip'

    # Auto download on Colab
    if IN_COLAB:
        try:
            colab_files.download(zip_file)
        except Exception:
            pass

    status = f'📦 Đã nén {len(audio_history)} file audio → all_audio.zip'
    return zip_file, status

# ── Gradio UI ──────────────────────────────────────────────────────
print('🎨 Khởi động giao diện Kiên Đoàn TTS — Ngọc Huyền...')
gr.close_all()

CSS = """
.gradio-container {
    max-width: 780px !important;
    margin: 0 auto !important;
    padding: 16px !important;
}
footer { display: none !important; }
.status-box {
    font-size: 14px;
    padding: 8px 12px;
    border-radius: 8px;
    background: #f0f4ff;
    border-left: 4px solid #4f46e5;
}
"""

THEME = gr.themes.Soft(primary_hue='indigo')

with gr.Blocks(title='Kiên Đoàn TTS — Ngọc Huyền') as demo:
    gr.Markdown(
        '# 🎙️ Kiên Đoàn TTS\n'
        '**Ngọc Huyền** · Giọng nữ thanh niên, tự nhiên, truyền cảm\n\n'
        '`Speed: 1.25x` · `Steps: 16` · `Output: MP3 192kbps` · `Auto Download: ON`'
    )

    with gr.Row():
        text_input = gr.Textbox(
            label='📝 Nhập văn bản',
            lines=5,
            placeholder='Nhập văn bản bạn muốn chuyển thành giọng nói...\n\nTách đoạn bằng 2 dòng trống để có khoảng nghỉ giữa các đoạn.'
        )

    with gr.Row():
        btn_generate = gr.Button('🎤 Tạo giọng nói', variant='primary', scale=3)
        btn_download_all = gr.Button('📦 Tải tất cả audio', variant='secondary', scale=1)

    # Status display
    status_text = gr.Textbox(
        label='📊 Trạng thái',
        interactive=False,
        value='Sẵn sàng! Nhập văn bản và bấm Tạo giọng nói.'
    )

    # Audio player — shows latest audio
    audio_output = gr.Audio(label='🎵 Audio mới nhất', type='numpy')

    # Download file — latest MP3
    file_output = gr.File(label='📥 File MP3 mới nhất (click để tải)')

    # Zip file output for download all
    zip_output = gr.File(label='📦 Zip tất cả audio', visible=False)

    # Wire up events
    btn_generate.click(
        fn=generate_voice,
        inputs=[text_input],
        outputs=[audio_output, file_output, status_text],
        concurrency_limit=1
    )

    btn_download_all.click(
        fn=download_all_audio,
        inputs=[],
        outputs=[zip_output, status_text]
    ).then(
        fn=lambda: gr.update(visible=True),
        outputs=[zip_output]
    )

# Launch with theme/css passed to launch() (Gradio 6.x fix)
demo.launch(
    server_name='0.0.0.0',
    share=True,
    theme=THEME,
    css=CSS,
    debug=True
)